# CAVIR processing pipeline

Reproduces `03_PREPROCESSED` of the CAVIR dataset from `02_PREPARSED`.

1. Set the path to the downloaded dataset below.
2. Run all cells. The output is written to `results/<timestamp>_nature_scientific/`.

The parameters are those of the published run (`03_PREPROCESSED/run_metadata.json`).

In [ ]:
from pathlib import Path

DATASET = Path("path/to/CAVIR_dataset")      # folder containing 01_RAW, 02_PREPARSED, participants.csv, ...
INPUT = Path("data")                         # working copy of the preparsed files (created below)
OUTPUT = Path("results")

## Participants and input data

The pipeline expects one folder per participant directly in the input folder, so the preparsed
files of both data-collection periods are copied into `INPUT`.

In [ ]:
import shutil
import pandas as pd

table = pd.read_csv(DATASET / "participants.csv", sep=";")
participants = {cohort: sorted(g["participant_id"]) for cohort, g in table.groupby("cohort")}

INPUT.mkdir(exist_ok=True)
for cohort, ids in participants.items():
    for pid in ids:
        shutil.copytree(DATASET / "02_PREPARSED" / cohort / pid, INPUT / pid, dirs_exist_ok=True)

{cohort: len(ids) for cohort, ids in participants.items()}

## Tasks and features

In [ ]:
tasks = [f"CAVIR_Start{n}" for n in range(1, 6)]     # working phase of tasks 1-5

metrics = {
    "eye_tracking": [
        "fixation_count", "fixation_rate_per_s", "avg_fixation_duration_ms", "fixation_total_ms",
        "fixation_time_fraction",
        "saccade_count", "saccade_rate_per_s", "avg_saccade_duration_ms", "saccade_total_ms",
        "saccade_time_fraction",
        "total_duration_s",
        "blink_count", "blink_rate_per_s", "avg_blink_duration_ms", "blink_time_prop",
        "pupil_mean_mm", "pupil_std_mm", "pupil_median_mm", "pupil_valid_fraction",
    ],
    "ecg": ["RMSSD", "SDNN", "pNN50", "NK_RMSSD", "NK_SDNN", "NK_pNN50"],   # heart rate (Polar RR intervals)
    "paas": ["score"],
}

## Parameters

Heart rate: the CAVIR data contain RR intervals only, so the raw-ECG options (R-peak detection,
band-pass, windowing) are not used. Eye tracking: samples are regularised to the 60 Hz tracker rate
before segmentation.

In [ ]:
ecg_params = {
    "polar_ticks_hz": 1024,                 # Polar RR intervals are given in 1/1024 s
    "rri_unit": "ticks",
    "rri_artifact_correction": "kubios",    # Lipponen & Tarvainen (2019), flagged beats are excluded
    "trim_start_sec": 5,
    "hr_outlier_range_bpm": [40, 200],
    # raw-ECG options (not used for RR-interval input)
    "sampling_rate_hz": 130,
    "rpeak_method": "neurokit",
    "rpeak_fallback": "heartpy",
    "neurokit": {"clean_method": "neurokit", "peak_method": "neurokit", "correct_artifacts": True},
    "window_sec": 40,
    "window_overlap": 0.5,
    "interpolation": "linear",
    "bandpass_hz": [5.0, 25.0],
}

et_params = {
    "sampling_rate_hz": 120,
    "segmentation": {
        "method": "ivt",
        "three_state": True,
        "use_hysteresis": True,
        "velocity_threshold_deg_s": 30,
        "fixation_on_threshold": 20,        # deg/s
        "fixation_off_threshold": 15,
        "saccade_on_threshold": 30,
        "saccade_off_threshold": 25,
        "min_fixation_duration_ms": 60,
        "min_saccade_duration_ms": 22,
        "min_mid_duration_ms": 80,
        "merge_fix_gap_ms": 30,
        "smooth_win_ms": 15,
    },
    "blinks": {"enabled": True, "min_duration_ms": 50, "max_duration_ms": 500},
    "missing_data_threshold": 0.2,
}

# modalities not processed for CAVIR (kept for the configuration format)
gsr_params = {"sampling_rate_hz": 51.2, "scr_min_amplitude": 0.01, "scl_window_sec": 10}
eeg_params = {
    "sampling_rate_hz": 250,
    "reference_channels": ["Fp1", "Fp2"],
    "montage": "standard",
    "frontal_channels": ["AF3", "F7", "F3", "F4", "F8", "AF4"],
    "parietal_channels": ["P7", "P8"],
}

## Run

In [ ]:
from modules.pipeline import run_pipeline

cfg = {
    "participants": participants,
    "tasks": tasks,
    "metrics": metrics,
    "params": {
        "ecg": ecg_params,
        "et": et_params,
        "gsr": gsr_params,
        "eeg": eeg_params,
        "head_movement": {},
        "paas": {"csv_path": str(DATASET / "paas.csv")},
        # recorded by design in both periods: missing files count as "missing", not "not recorded"
        "qc": {"expected_modalities": {c: ["eye_tracking", "ecg", "paas"] for c in participants}},
    },
    "export_mode": "combined",
    "file_structure": "PTM",          # results/<run>/<participant>/<task>/<modality>
    "label": "nature_scientific",
    "input_path": str(INPUT),
    "output_path": str(OUTPUT),
}

run_pipeline(cfg)